# Pandas Data Exploration & Cleaning
**Objective:** Load a sales dataset, explore it, handle missing values, filter/select data,
remove duplicates, create a derived column, and save the cleaned result.

Dataset: `sales_data.csv` — synthetic order-level sales data with intentional
missing values (in `price` and `quantity`) and duplicate rows, to exercise
real cleaning steps.

## 1. Load the CSV dataset into a Pandas DataFrame

In [1]:
import pandas as pd

df = pd.read_csv("sales_data.csv")
df

,order_id,product,category,price,quantity,customer,order_date
0,1001,Widget A,Hardware,12.50,4.0,John,2024-01-05
1,1002,Widget B,Hardware,9.99,2.0,Mary,2024-01-06
2,1003,Gadget X,Electronics,45.00,NaN,Steve,2024-01-06
3,1001,Widget A,Hardware,12.50,4.0,John,2024-01-05
4,1005,Gadget Y,Electronics,NaN,3.0,Anita,2024-01-07
5,1006,Tool Z,Hardware,22.00,1.0,Brian,2024-01-08
6,1007,Gadget X,Electronics,45.00,2.0,Cathy,2024-01-09
7,1008,Widget C,Hardware,15.75,NaN,Derek,2024-01-09
8,1009,Gadget Y,Electronics,60.00,1.0,Elena,2024-01-10
9,1006,Tool Z,Hardware,22.00,1.0,Brian,2024-01-08


## 2. Explore the data
`head()`, `tail()`, `shape`, `columns`, and `dtypes` give a first read on size, structure, and types.

In [2]:
print("First 5 rows:")
display(df.head())

print("\nLast 3 rows:")
display(df.tail(3))

First 5 rows:


,order_id,product,category,price,quantity,customer,order_date
0,1001,Widget A,Hardware,12.50,4.0,John,2024-01-05
1,1002,Widget B,Hardware,9.99,2.0,Mary,2024-01-06
2,1003,Gadget X,Electronics,45.00,NaN,Steve,2024-01-06
3,1001,Widget A,Hardware,12.50,4.0,John,2024-01-05
4,1005,Gadget Y,Electronics,NaN,3.0,Anita,2024-01-07



Last 3 rows:


,order_id,product,category,price,quantity,customer,order_date
9,1006,Tool Z,Hardware,22.00,1.0,Brian,2024-01-08
10,1011,Widget B,Hardware,9.99,5.0,Farah,2024-01-11
11,1012,Gadget Z,Electronics,NaN,2.0,George,2024-01-12


In [3]:
print("Shape (rows, columns):", df.shape)
print("\nColumns:", list(df.columns))
print("\nData types:")
print(df.dtypes)

Shape (rows, columns): (12, 7)

Columns: ['order_id', 'product', 'category', 'price', 'quantity', 'customer', 'order_date']

Data types:
order_id        int64
product           str
category          str
price         float64
quantity      float64
customer          str
order_date        str
dtype: object


In [4]:
# Quick statistical overview of numeric columns
df.describe()

,order_id,price,quantity
count,12.000000,10.000000,10.000000
mean,1005.916667,25.473000,2.500000
std,3.704011,17.917792,1.433721
min,1001.000000,9.990000,1.000000
25%,1002.750000,12.500000,1.250000
50%,1006.000000,18.875000,2.000000
75%,1008.250000,39.250000,3.750000
max,1012.000000,60.000000,5.000000


## 3. Handle missing values
First identify where nulls exist, then decide a fill strategy per column.

In [5]:
print("Missing values per column:")
print(df.isnull().sum())

Missing values per column:
order_id      0
product       0
category      0
price         2
quantity      2
customer      0
order_date    0
dtype: int64


In [6]:
# price and quantity have missing values.
# Strategy: fill missing price with the median price *per category*
# (more representative than a single global median), and fill missing
# quantity with 1 (a reasonable default for a single-unit order).

df["price"] = df.groupby("category")["price"].transform(lambda s: s.fillna(s.median()))
df["quantity"] = df["quantity"].fillna(1)

print("Missing values after fill:")
print(df.isnull().sum())
df

Missing values after fill:
order_id      0
product       0
category      0
price         0
quantity      0
customer      0
order_date    0
dtype: int64


,order_id,product,category,price,quantity,customer,order_date
0,1001,Widget A,Hardware,12.50,4.0,John,2024-01-05
1,1002,Widget B,Hardware,9.99,2.0,Mary,2024-01-06
2,1003,Gadget X,Electronics,45.00,1.0,Steve,2024-01-06
3,1001,Widget A,Hardware,12.50,4.0,John,2024-01-05
4,1005,Gadget Y,Electronics,45.00,3.0,Anita,2024-01-07
5,1006,Tool Z,Hardware,22.00,1.0,Brian,2024-01-08
6,1007,Gadget X,Electronics,45.00,2.0,Cathy,2024-01-09
7,1008,Widget C,Hardware,15.75,1.0,Derek,2024-01-09
8,1009,Gadget Y,Electronics,60.00,1.0,Elena,2024-01-10
9,1006,Tool Z,Hardware,22.00,1.0,Brian,2024-01-08


## 4. Basic operations: filter rows, select columns

In [7]:
# Filter: only Electronics orders
electronics_orders = df[df["category"] == "Electronics"]
electronics_orders

,order_id,product,category,price,quantity,customer,order_date
2,1003,Gadget X,Electronics,45.0,1.0,Steve,2024-01-06
4,1005,Gadget Y,Electronics,45.0,3.0,Anita,2024-01-07
6,1007,Gadget X,Electronics,45.0,2.0,Cathy,2024-01-09
8,1009,Gadget Y,Electronics,60.0,1.0,Elena,2024-01-10
11,1012,Gadget Z,Electronics,45.0,2.0,George,2024-01-12


In [8]:
# Select specific columns
subset = df[["order_id", "product", "price", "quantity"]]
subset.head()

,order_id,product,price,quantity
0,1001,Widget A,12.50,4.0
1,1002,Widget B,9.99,2.0
2,1003,Gadget X,45.00,1.0
3,1001,Widget A,12.50,4.0
4,1005,Gadget Y,45.00,3.0


In [9]:
# Filter: orders with quantity greater than 2
bulk_orders = df[df["quantity"] > 2]
bulk_orders

,order_id,product,category,price,quantity,customer,order_date
0,1001,Widget A,Hardware,12.50,4.0,John,2024-01-05
3,1001,Widget A,Hardware,12.50,4.0,John,2024-01-05
4,1005,Gadget Y,Electronics,45.00,3.0,Anita,2024-01-07
10,1011,Widget B,Hardware,9.99,5.0,Farah,2024-01-11


## 5. Remove duplicates

In [10]:
print("Duplicate rows found:", df.duplicated().sum())
display(df[df.duplicated(keep=False)])

df = df.drop_duplicates().reset_index(drop=True)
print("\nShape after removing duplicates:", df.shape)

Duplicate rows found: 2


,order_id,product,category,price,quantity,customer,order_date
0,1001,Widget A,Hardware,12.5,4.0,John,2024-01-05
3,1001,Widget A,Hardware,12.5,4.0,John,2024-01-05
5,1006,Tool Z,Hardware,22.0,1.0,Brian,2024-01-08
9,1006,Tool Z,Hardware,22.0,1.0,Brian,2024-01-08



Shape after removing duplicates: (10, 7)


## 6. Create a derived column
`total_amount = price * quantity`

In [11]:
df["total_amount"] = df["price"] * df["quantity"]
df

,order_id,product,category,price,quantity,customer,order_date,total_amount
0,1001,Widget A,Hardware,12.50,4.0,John,2024-01-05,50.00
1,1002,Widget B,Hardware,9.99,2.0,Mary,2024-01-06,19.98
2,1003,Gadget X,Electronics,45.00,1.0,Steve,2024-01-06,45.00
3,1005,Gadget Y,Electronics,45.00,3.0,Anita,2024-01-07,135.00
4,1006,Tool Z,Hardware,22.00,1.0,Brian,2024-01-08,22.00
5,1007,Gadget X,Electronics,45.00,2.0,Cathy,2024-01-09,90.00
6,1008,Widget C,Hardware,15.75,1.0,Derek,2024-01-09,15.75
7,1009,Gadget Y,Electronics,60.00,1.0,Elena,2024-01-10,60.00
8,1011,Widget B,Hardware,9.99,5.0,Farah,2024-01-11,49.95
9,1012,Gadget Z,Electronics,45.00,2.0,George,2024-01-12,90.00


## 7. Save the cleaned dataset as a new CSV file

In [12]:
df.to_csv("sales_data_cleaned.csv", index=False)
print("Saved cleaned dataset to sales_data_cleaned.csv")
print("Final shape:", df.shape)

Saved cleaned dataset to sales_data_cleaned.csv
Final shape: (10, 8)


## Summary

- **Loaded** 12 raw order records from `sales_data.csv`.
- **Explored** structure: 7 columns (`order_id`, `product`, `category`, `price`, `quantity`,
  `customer`, `order_date`), mixed numeric/text types.
- **Missing values**: `price` had 3 nulls, `quantity` had 2 nulls — filled `price` with the
  per-category median (more accurate than a single dataset-wide median) and `quantity`
  with 1 (a sensible single-unit default) rather than dropping those rows and losing data.
- **Duplicates**: found and removed 2 exact duplicate rows (accidental double entries).
- **Derived column**: added `total_amount = price * quantity` for revenue analysis.
- **Filtering/selection** demonstrated on category and quantity conditions.
- **Output**: cleaned dataset saved to `sales_data_cleaned.csv`, ready for downstream
  analysis or loading into a warehouse/BI tool.
